# 2장. CAE 학습 데이터 준비와 ANSYS 결과 변환

## 개요

**학습 목표:** ANSYS 해석 데이터 준비 과정 이해 / RST의 메시/응력/변위를 VTU로 정리

| 순서 | 알아볼 내용 |
|---|---|
| 1. 학습 데이터 구성 | 모델의 입력/정답과 해석 케이스 |
| 2. DOE와 ANSYS 해석 데이터 준비 | 조건 선정 / 설계점 해석 / RST 보관 |
| 3. 해석 데이터 구조 | RST/VTU, 메시, 파일 대응과 단위 |
| 4. PyDPF의 역할 | Python에서 RST를 읽는 도구 |
| 5 ~ 6. 실행 준비와 RST 확인 | 경로/케이스 선택, 실제 메시와 결과 조회 |
| 7 ~ 8. 결과 정리와 저장 | ID 정렬, 응력 처리, 단위 변환과 VTU 저장 |
| 9. 변환 결과 검증 | 결과 그림/최대값 대조/제공 자료 확인 |
| 10. 다음 장에서 사용할 자료 | VTU 항목을 모델 입력/정답으로 연결 |


## 5. 실행환경과 입출력 경로 설정


### 02장 / 셀 15 / Notebook


In [ ]:
# 1. 파일/숫자 배열/메시/결과 조회에 사용할 도구를 준비합니다.
from pathlib import Path
import csv
import json
import os
import runpy
import sys
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from ansys.dpf import core as dpf

# 2. 현재 위치와 상위 폴더에서 교육 폴더를 찾습니다.
working_dir = Path.cwd().resolve()
candidates = []
for parent in (working_dir, *working_dir.parents):
    candidates.extend([parent / "preparation" / "convert_rst.py",
                       parent / "education" / "preparation" / "convert_rst.py"])
converter_path = next((path for path in candidates if path.is_file()), None)
if converter_path is None:
    raise FileNotFoundError("프로젝트 루트 또는 education 하위 폴더에서 실행하세요.")
education_dir = converter_path.parent.parent
data_dir = education_dir / "data"
# 3. 제공된 변환 기능을 연결하고 ANSYS 설치 경로를 확인합니다.
converter = runpy.run_path(str(converter_path), run_name="conversion_notebook_helper")
ansys_path = os.environ.get("AWP_ROOT252", "C:/Program Files/ANSYS Inc/v252")
# 설치 위치가 다르면 위 값을 실제 ANSYS 2025 R2 경로로 수정한다.
if not ansys_path or not Path(ansys_path).is_dir():
    raise FileNotFoundError("ANSYS 2025 R2 설치 경로를 AWP_ROOT252로 설정하세요.")
print("교육자료:", education_dir)
print("ANSYS 2025 R2:", ansys_path)

print("커널 Python:", sys.executable)
# 4. 이번 실행의 출력 폴더를 만들고 제공 파일의 식별값을 기록합니다.
RUN_NAME = os.environ.get("EDUCATION_CONVERSION_RUN", "student_beginner_run_01")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME은 경로 구분자 없는 폴더 이름으로 지정하세요.")
output_dir = (education_dir / "runs/conversion" / RUN_NAME).resolve()
assert not output_dir.is_relative_to(data_dir.resolve())
output_dir.mkdir(parents=True, exist_ok=True)
provided_hashes = {p.name: converter["sha256"](p) for p in (data_dir / "vtu").iterdir() if p.is_file()}
print("실습 출력:", output_dir)


## 6. 대표 케이스의 RST 데이터 확인


### 6.1 조건표와 RST 대응 확인


### 02장 / 셀 18 / Notebook


In [ ]:
# 1. 조건과 파일 대응표를 읽어 10개 케이스를 출력합니다.
records, references = converter["load_case_records"]()
print("case_id     Workbench DP  thickness_mm  force_N")
for record in records:
    print(f"{record['case_id']}  {record['workbench_design_point']:12s}  "
          f"{float(record['thickness_mm']):12.3f}  {float(record['force_N']):7.3f}")
# 2. 살펴볼 대표 케이스를 선택하고 RST 위치를 확인합니다.
case_id = "case_001"
record = next(row for row in records if row["case_id"] == case_id)
rst_path = data_dir / record["rst_file"]
print("대표 결과:", rst_path.name, "존재:", rst_path.is_file())

# 3. 케이스별 두께/하중이 조건표와 일치하는지 확인합니다.
with (data_dir / "lhs_doe.csv").open(encoding="utf-8-sig", newline="") as stream:
    doe_rows = {row["case_id"]: row for row in csv.DictReader(stream)}
assert len(records) == len(doe_rows) == 10
for row in records:
    assert all(float(row[key]) == float(doe_rows[row["case_id"]][key])
               for key in ["thickness_mm", "force_N"])
# 4. 이 케이스의 ANSYS 해석 기준 최대값을 읽습니다.
reference = references[record["workbench_design_point"]]
print("ANSYS 해석 기준 최대응력(MPa):", float(reference["P4"]) / 1e6)
print("ANSYS 해석 기준 최대 변위 크기(mm):", float(reference["P3"]) * 1000)
print("조건표와 RST 대응: PASS")


### 6.2 PyDPF로 메시와 해석 결과 조회


### 02장 / 셀 21 / Notebook


In [ ]:
# 1. RST 조회를 처리할 DPF 서버를 시작합니다.
server = dpf.start_local_server(ansys_path=ansys_path, as_global=False,
                                context=dpf.AvailableServerContexts.entry, timeout=30)
try:
    # 2. 대표 RST를 열고 메시와 마지막 저장 결과 단계를 찾습니다.
    model = dpf.Model(str(rst_path), server=server)
    mesh = model.metadata.meshed_region
    result_times = np.array(model.metadata.time_freq_support.time_frequencies.data)
    selected_set = int(result_times.size)
    args = dict(time_scoping=[selected_set], data_sources=model.metadata.data_sources, server=server)
    # 3. 그 단계의 변위와 응력 Field를 읽습니다.
    displacement_field = dpf.operators.result.displacement(**args).outputs.fields_container()[0]
    stress_field = dpf.operators.result.stress(**args, bool_rotate_to_global=True).outputs.fields_container()[0]
    # 4. 메시 크기/단위/배열 모양/대상 ID를 확인합니다.
    print("DPF server:", server.version)
    print("노드 / 요소:", mesh.nodes.n_nodes, "/", mesh.elements.n_elements)
    print("좌표 단위:", mesh.unit)
    print("저장 결과 시간:", result_times.tolist(), "선택 set:", selected_set)
    print("변위:", displacement_field.location, displacement_field.unit, displacement_field.data.shape)
    print("응력:", stress_field.location, stress_field.unit, stress_field.data.shape)
    print("격자 노드 ID 예시:", list(mesh.nodes.scoping.ids[:5]))
    print("변위 Field scoping ID 예시:", list(displacement_field.scoping.ids[:5]))
    print("변위 Field 값 예시:", np.array(displacement_field.data[:5]))
    print("응력 Field scoping 요소 ID 예시:", list(stress_field.scoping.ids[:5]))
    print("응력 scoping 대상 수:", len(stress_field.scoping.ids))
# 5. 조회가 끝나거나 오류가 나면 서버를 종료합니다.
finally:
    server.shutdown()


## 7. 노드 정렬과 응력 처리


### 7.1 노드 ID에 맞춰 결과 연결하기


### 02장 / 셀 25 / Notebook


In [ ]:
# 1. 좌표 쪽의 목표 ID와 결과 쪽의 ID/값을 준비합니다.
target_ids = np.array([103, 101, 102])
source_ids = np.array([101, 103, 102])
source_values = np.array([10.0, 20.0, 30.0])
# 2. ID를 대응시켜 결과를 목표 순서로 배치합니다.
order = converter["index_by_id"](target_ids, source_ids)
aligned_values = source_values[order]
# 3. 배치한 값이 각 ID의 결과와 맞는지 확인합니다.
print("목표 ID:", target_ids.tolist())
print("원래 값:", source_values.tolist())
print("정렬한 값:", aligned_values.tolist())
assert aligned_values.tolist() == [20.0, 10.0, 30.0]


### 7.3 참고: 응력 계산 순서가 중요한 이유 — 선택 예제


### 02장 / 셀 28 / Notebook


In [ ]:
# 성분 순서: SX, SY, SZ, SXY, SYZ, SXZ / 단위 MPa
# 1. 계산 순서를 비교할 두 응력 상태를 준비합니다.
element_tensors = np.array([[100.0, 0, 0, 0, 0, 0],
                            [0, 100.0, 0, 0, 0, 0]])
vm = converter["von_mises"]
# 2. 성분 평균 후 계산과 등가응력 계산 후 평균을 각각 수행합니다.
tensor_first = float(vm(element_tensors.mean(axis=0)))
scalar_first = float(vm(element_tensors).mean())
# 3. 두 결과가 달라지는지 비교합니다.
print("성분을 평균한 뒤 von Mises:", tensor_first, "MPa")
print("von Mises를 계산한 뒤 평균:", scalar_first, "MPa")


## 8. RST의 VTU 변환


### 8.1 단위를 맞추고 대표 케이스 저장하기


### 02장 / 셀 31 / Notebook


In [ ]:
# 1. DPF 서버를 시작하고 실제 RST 변환 함수를 실행합니다.
server = dpf.start_local_server(ansys_path=ansys_path, as_global=False,
                                context=dpf.AvailableServerContexts.entry, timeout=30)
try:
    summary, report = converter["convert_case"](record, references[record["workbench_design_point"]], server,
                                                 output_dir=output_dir)
# 2. 변환 후 서버를 종료합니다.
finally:
    server.shutdown()
# 3. 생성된 VTU를 읽어 노드 결과의 크기/단위/유효성을 확인합니다.
generated_vtu = (data_dir / summary["vtu_file"]).resolve()
assert generated_vtu.parent == output_dir
grid = pv.read(generated_vtu)
for field in ["node_id", "von_mises_stress", "displacement", "displacement_magnitude"]:
    print(field, grid.point_data[field].shape)
assert grid["displacement"].shape == (grid.n_points, 3)
assert np.isfinite(grid["von_mises_stress"]).all()
assert np.unique(grid["node_id"]).size == grid.n_points
print("VTU 단위:", grid.field_data["coordinate_unit"].tolist(),
      grid.field_data["stress_unit"].tolist(), grid.field_data["displacement_unit"].tolist())

# 4. 제공 파일이 보존됐는지 확인하고 새 파일의 위치를 출력합니다.
assert all(converter["sha256"](data_dir / "vtu" / name) == digest
           for name, digest in provided_hashes.items())
print("생성 VTU:", generated_vtu)
print("검증 JSON:", output_dir / f"{case_id}_validation.json")
print("배포된 VTU/검증 기록 보존: PASS")


### 8.2 저장된 VTU의 구성 확인


### 02장 / 셀 34 / Notebook


In [ ]:
# 1. 저장된 좌표와 점별/요소별/파일 전체 자료의 크기를 확인합니다.
print("메시:", grid.n_points, "points /", grid.n_cells, "cells")
print("points:", grid.points.shape)
for label, arrays in [("point_data", grid.point_data), ("cell_data", grid.cell_data),
                      ("field_data", grid.field_data)]:
    print("\n" + label)
    for name in arrays:
        print(name, np.asarray(arrays[name]).shape)
# 2. 첫 요소의 점 인덱스와 원래 노드 ID를 대조합니다.
assert np.all(grid.celltypes == 24)
first_cell = grid.cells.reshape(-1, 11)[0]
assert first_cell[0] == 10
indices = first_cell[1:]
print("첫 요소의 points 인덱스:", indices.tolist())
print("같은 요소의 해석 노드 ID:", grid.point_data["node_id"][indices].tolist())
# 3. 저장된 케이스 이름과 두께/하중을 확인합니다.
print("case / thickness / force:", grid.field_data["case_id"].tolist(),
      grid.field_data["thickness_mm"].tolist(), grid.field_data["force_N"].tolist())


## 9. 변환 결과 검증


### 9.1 ANSYS 해석 기준 최대값 대조


### 02장 / 셀 37 / Notebook


In [ ]:
# 1. 변환 결과와 ANSYS 해석 최대값의 차이를 확인합니다.
print("항목                 변환 결과       ANSYS 해석      차이(%)")
print(f"최대응력(MPa)       {report['max_stress_mpa']:.9f}  "
      f"{report['workbench_max_stress_mpa']:.9f}  {report['max_stress_difference_percent']:.3e}")
print(f"최대 변위 크기(mm)        {report['max_displacement_mm']:.9f}  "
      f"{report['workbench_max_displacement_mm']:.9f}  {report['max_displacement_difference_percent']:.3e}")
# 2. 전체 노드 응력과 저장/재로딩 검사 결과를 확인합니다.
print("코너 / 중간 노드:", report["corner_nodes"], "/", report["midside_nodes"])
print("노드 응력 유한값:", report["all_fields_finite"])
print("전체 노드 응력 커버리지:", report["complete_stress_coverage"])
print("VTU 저장/재로딩 확인:", report["vtu_round_trip_verified"])


### 9.2 체적/표면 메시와 결과 시각화


### 02장 / 셀 40 / Notebook


In [ ]:
# 1. 전체 체적 메시에서 그림에 표시할 표면을 추출합니다.
surface = grid.extract_surface(algorithm="dataset_surface", nonlinear_subdivision=1)
print("체적 메시:", grid.n_points, "nodes /", grid.n_cells, "elements")
print("추출 표면:", surface.n_points, "points /", surface.n_cells, "faces")
# 2. 응력과 변위 크기를 각각 색상으로 표시합니다.
fig = plt.figure(figsize=(12, 5))
for index, (name, unit) in enumerate([("von_mises_stress", "MPa"),
                                     ("displacement_magnitude", "mm")], start=1):
    axis = fig.add_subplot(1, 2, index, projection="3d")
    artist = axis.scatter(surface.points[:, 0], surface.points[:, 1], surface.points[:, 2],
                          c=surface.point_data[name], s=3, cmap="viridis")
    axis.set(xlabel="X (mm)", ylabel="Y (mm)", zlabel="Z (mm)", title=f"{case_id}: {name}")
    axis.set_box_aspect(np.maximum(np.ptp(surface.points, axis=0), 1e-6))
    fig.colorbar(artist, ax=axis, shrink=0.6, label=unit)
# 3. 그림을 실습 폴더에 저장하고 화면에 표시합니다.
fig.tight_layout()
figure_path = output_dir / f"{case_id}_fields.png"
fig.savefig(figure_path, dpi=140)
plt.show()
plt.close(fig)
print("생성 그림:", figure_path)


### 9.3 제공 데이터의 검증 기록 확인


### 02장 / 셀 43 / Notebook


In [ ]:
# 1. 강사가 제공한 10개 결과의 묶음 검증 기록과 요약표를 읽습니다.
provided_dir = data_dir / "vtu"
batch_path = provided_dir / "batch_validation.json"
if not batch_path.is_file():
    raise FileNotFoundError("배포된 data/vtu의 검증 파일을 확인하세요. 새 결과는 실습 출력 폴더에 저장합니다.")
batch = json.loads(batch_path.read_text(encoding="utf-8"))
with (provided_dir / "conversion_summary.csv").open(encoding="utf-8-sig", newline="") as stream:
    converted = list(csv.DictReader(stream))
# 2. 케이스 수/조건/개별 검사 기록/파일 식별값을 확인합니다.
assert batch["all_passed"] and batch["cases_converted"] == 10 and len(converted) == 10
assert {row["case_id"] for row in converted} == set(doe_rows)
for row in converted:
    case_report = json.loads((provided_dir / f"{row['case_id']}_validation.json").read_text(encoding="utf-8"))
    assert case_report["validation"] == "PASS"
    assert converter["sha256"](provided_dir / f"{row['case_id']}.vtu") == case_report["vtu_sha256"]
    assert all(float(row[key]) == float(doe_rows[row["case_id"]][key])
               for key in ["thickness_mm", "force_N"])
# 3. 전체 통과 여부와 케이스별 결과 요약을 출력합니다.
print("제공된 전체 변환:", batch["cases_converted"], "개, 전체 통과:", batch["all_passed"])
print("case_id      nodes  elements  max_stress_MPa  max_displacement_mm")
for row in converted:
    print(f"{row['case_id']}  {int(row['n_nodes']):6d}  {int(row['n_elements']):8d}  "
          f"{float(row['max_stress_mpa']):14.6f}  {float(row['max_displacement_mm']):19.6f}")


### 9.4 전체 케이스 재변환 — 선택 실습


### 02장 / 셀 45 / Console 01


```bat
python education/preparation/convert_rst.py --case all --output-dir education/runs/conversion/student_beginner_all_01
```
